# Data Scraper

## Import Libraries and Set Global Settings

In [ ]:
import time
import random
import re
import pandas as pd
from datetime import datetime
from playwright.sync_api import sync_playwright, TimeoutError as PlaywrightTimeoutError

# Define the base website URL
BASE_URL = "https://www.autotrader.co.uk"
# Initialize the base search filter URL
BASE_SEARCH_URL = "https://www.autotrader.co.uk/car-search?advertising-location=at_cars&channel=cars&fuel-type=Electric&homeDeliveryAdverts=include&make=&postcode=SW1A%201AA&sort=relevance&year-to=2026"

# Set the target number of cars to scrape
TARGET_CARS = 50

# Generate a unique timestamp for the output filename
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
# Define the output path for the scraped dataset
DATA_PATH = f'../data/processed/auto_trader_uk_scraped_cars_{TIMESTAMP}.csv'

# Initialize list for car record dictionaries
scraped_cars = []
# Initialize set for unique car links
car_links = set()

## Web Scraping

In [ ]:
# Initialize the playwright browser engine
with sync_playwright() as p:
    # Launch a chromium browser instance in headless mode
    browser = p.chromium.launch(headless=True) 
    # Define a custom user agent string
    user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    # Create a new browser context with the user agent
    context = browser.new_context(user_agent=user_agent)
    # Open a new browser page
    page = context.new_page()

    # Initialize the page number counter
    page_num = 1
    
    # Iterate through search pages until target link count is reached
    while len(car_links) < TARGET_CARS:
        # Construct the paginated search URL
        paginated_url = f"{BASE_SEARCH_URL}&page={page_num}"
        # Output current page progress
        print(f"Loading search feed: Page {page_num}...")
        
        try:
            # Navigate to the paginated search URL
            page.goto(paginated_url, timeout=15000)
        except PlaywrightTimeoutError:
            # Output timeout warning and stop pagination
            print(f"Timeout on page {page_num}. Stopping pagination...")
            break
        
        # Handle cookie consent on the initial page load
        if page_num == 1:
            try:
                # Identify the cookie acceptance button
                accept_button = page.locator('button:has-text("Accept All"), button:has-text("Accept all cookies")').first
                # Check if the accept button is visible
                if accept_button.is_visible(timeout=5000):
                    # Click the cookie acceptance button
                    accept_button.click()
                    # Output confirmation of cookie acceptance
                    print("Accepted cookies.")
                    # Introduce a randomized delay
                    time.sleep(random.uniform(1.0, 3.0))
            except Exception:
                # Output notice if no cookie banner is found
                print("No cookie banner found. Proceeding...")
            
        # Introduce a randomized delay to mimic human behavior
        time.sleep(random.uniform(1.5, 3.5))
        
        # Identify search listing title elements
        elements = page.locator('a[data-testid="search-listing-title"]').all()
        
        # Check if listing elements were found on the page
        if not elements:
            # Output end of results notice
            print(f"No more cars found. Reached end of search results.")
            break
            
        # Store the current link count for comparison
        initial_count = len(car_links)
        # Iterate through identified elements
        for el in elements:
            # Extract the href attribute from the element
            href = el.get_attribute('href')
            if href:
                # Clean the URL and add the base domain
                clean_url = BASE_URL + href.split('?')[0]
                # Add the clean URL to the links set
                car_links.add(clean_url)
                
        # Output current link extraction progress
        print(f"Found {len(car_links)} unique car links...")
        
        # Check if the link count failed to increase
        if len(car_links) == initial_count:
            # Output warning for pagination caps
            print("No new links found. AutoTrader may be capping results.")
            break
            
        # Increment the page number counter
        page_num += 1
        
    # Truncate the link list to the target count
    final_links = list(car_links)[:TARGET_CARS]

    # Output notice for individual detail extraction
    print(f"Beginning extraction of {len(final_links)} cars...")

    # Iterate through the final list of car links
    for link in final_links:
        try:
            # Navigate to the individual listing page
            page.goto(link)
            # Introduce a randomized delay
            time.sleep(random.uniform(1.5, 3.0))
            
            # Extract the header text from the page
            h1_text = page.locator('h1').inner_text(timeout=5000)
            # Split the header text into parts
            parts = h1_text.split(' ', 1)
            # Identify the car make and format to lowercase
            make = parts[0].strip().lower()
            # Identify the car model or set as unknown
            model = parts[1].strip().lower() if len(parts) > 1 else 'unknown'
            
            # Extract the price text from the page
            price_text = page.locator('[data-testid="advert-price"]').inner_text()
            # Use regex to convert price text to float
            price = float(re.sub(r'[^\d.]', '', price_text))
            
            # Extract the mileage text using xpath
            mileage_text = page.locator('xpath=//p[text()="Mileage"]/following-sibling::p').inner_text()
            # Use regex to convert mileage text to float
            mileage = float(re.sub(r'[^\d.]', '', mileage_text))
            
            # Extract the registration text using xpath
            reg_text = page.locator('xpath=//p[text()="Registration"]/following-sibling::p').inner_text()
            # Use regex to find the registration year
            year_match = re.search(r'\d{4}', reg_text)
            # Assign the matched year or default to 2020
            year = int(year_match.group(0)) if year_match else 2020
            
            # Extract the gearbox text using xpath
            transmission_text = page.locator('xpath=//p[text()="Gearbox"]/following-sibling::p').inner_text()
            # Format the transmission text to lowercase
            transmission = transmission_text.strip().lower()
            
            # Construct the age feature based on current year 2026
            age = 2026 - year 
            
            # Output extraction summary for the current car
            print(f"Scraped: {make.title()} {model.title()} - £{price}")
            
            # Append the extracted car data to the results list
            scraped_cars.append({
                'country': 'uk',
                'make': make,
                'model': model,
                'year': year,
                'age': age,
                'mileage': mileage,
                'transmission': transmission,
                'fuelType': 'electric', 
                'mpg': 0.0,             
                'engineSize': 0.0,      
                'price': price
            })
            
        except PlaywrightTimeoutError:
            # Output timeout warning for the car page
            print(f"Timeout on car page: {link}")
            continue
        except Exception as e:
            # Output extraction failure notice
            print(f"Failed to scrape {link}: {str(e)}")
            continue
                
    # Close the browser instance
    browser.close()

## Export Scrapped Data

In [ ]:
# Check if the results list is not empty
if scraped_cars:
    # Initialize a dataframe from the results list
    df_new = pd.DataFrame(scraped_cars)
    
    # Initialize ordered_columns list for consistency
    ordered_columns = ['country', 'make', 'model', 'year', 'age', 'mileage', 'transmission', 'fuelType', 'mpg', 'engineSize', 'price']
    
    # Reorder the dataframe columns
    df_new = df_new[ordered_columns]
    
    # Save the dataframe to a CSV file
    df_new.to_csv(NEW_DATA_PATH, index=False)
    
    # Output final save confirmation
    print(f"Successfully saved {len(df_new)} cars to a new file: {NEW_DATA_PATH}")
else:
    # Output notice for empty results
    print("No data scraped to save.")